Help : https://cengel.github.io/R-spatial/spatialops.html

In [ ]:
system("conda install -y conda-forge::r-rcpp conda-forge::openssl conda-forge::r-sf conda-forge::r-terra conda-forge::r-ncdf4")
system("conda install -y conda-forge::r-r.utils conda-forge::r-tidyverse conda-forge::libgdal-hdf5 conda-forge::r-ggplot2")
system("conda install -y conda-forge::r-lubridate conda-forge::r-rcolorbrewer conda-forge::r-lattice conda-forge::r-png r::r-raster")
system("conda install -y conda-forge::r-cluster conda-forge::r-dendextend bioconda::r-bioregion")
system("conda install -y conda-forge::r-geojsonio")
system("conda install -y conda-forge::r-hrbrthemes conda-forge::r-viridis")

In [ ]:
# library(jsonlite) 

library(sf)
library(terra)
library(geojsonio)
library(R.utils)
library(utils)
library(tidyverse) # because who can live without the tidyverse?
library(dplyr)

library(ncdf4)
library(lubridate) # lubridate: operate on date and times data
library(lattice) # lattice : visualization system for typical graphics

library(data.table)

library(dendextend)
library(ggplot2)
library(RColorBrewer) # RColorBrewer: create colour palettes for thematic maps
library(viridis)
library(hrbrthemes)

In [ ]:
library(cluster)
library(bioregion)

In [ ]:
# SET DIRECTORIES
workdir <- getwd()
dataDir <- paste(workdir,"Data",sep = "/")
outputDir <- paste(workdir,"outputs/collection",sep = "/")
scriptDir <- paste(workdir,"scripts",sep = "/")

In [ ]:
# GENERAL FUNCTIONS

# ==============================================================================
# CREATE REPERTORY
# ==============================================================================
create.directory <- function(name.directory){
    ifelse(!dir.exists(file.path(name.directory)),
        dir.create(file.path(name.directory)),
        "Directory Exists")
}

# ==============================================================================
# DOWNLOAD FILENAME
# ==============================================================================
download.filename <- function(filename, url){
    options(timeout = 600)  # 10 minutes
    
    if(file.exists(filename)){
        cat(filename, "is (are) already in your repertory.")
    } else {
        download.file(url, filename, mode = "wb")
        print('File Downloaded')
    }
}

# ==============================================================================
# UNZIP FILENAME
# ==============================================================================
unzip.file <- function(filename, type = "gz"){
    filename.length <- nchar(filename)
    print(filename, filename.length)

    start <- 1

    if(type == "gz"){
        # Case Gunzip
        end <- filename.length-3
    }
    else{
        # Case Unzip
        end <- filename.length-4
    }

    unzip.filename <- substr(filename,start, end)
    print(unzip.filename)

    
    # Case gunzip
    if(!file.exists(unzip.filename)){
        if(type == "gz"){
            R.utils::gunzip(filename, overwrite=FALSE, remove=TRUE, BFR.SIZE=1e+07)
            }
        else{
            utils::unzip(filename, overwrite=FALSE )
            }
        cat(filename, "successfully unzipped!")
    }

    return(unzip.filename)
}

# ==============================================================================
# EXPORT AS A CSV 
# ==============================================================================
export.csv <- function(directory = outputDir, table.output, filename){
    table_filename <- paste(directory,filename, sep="/")
    write.table(table.output, table_filename, row.names=TRUE, col.names = TRUE, sep=",")
    print(paste(filename,"output exported.", sep = " "))
    }

# ==============================================================================
# MOVE TO DATA DIRECTORY 
# ==============================================================================
move.file <- function(filename, new.path){
    new.filename <- paste(new.path, filename, sep = "/")
    file.rename(from=filename, to=new.filename)
    return(new.filename)
}


In [ ]:
gdb_path_zip <- "Geomorphology.gdb.zip"

download.filename(url = "https://d28rz98at9flks.cloudfront.net/102441/Geomorphology.gdb.zip",
                 filename = gdb_path_zip) # https://data.gov.au/data/dataset/geomorphic-features-of-the-antarctic-margin-and-southern-ocean-20126/resource/cedf76d9-4390-4080-b9d0-84c57bd1c7b8

gdb_path <- unzip.file(gdb_path_zip, type = "zip")

# gdb_path <- move.file(gdb_path, paste(dataDir,"Geomorphology.gdb",sep = "/"))


In [ ]:
#Correct unzipped path
gdb_path <- unlist(strsplit(gdb_path, "/"))[[length( unlist(strsplit(gdb_path, "/")) )]]
print(gdb_path)

In [ ]:
# httpsjjj://gis.stackexchange.com/questions/426282/load-gdb-directory-into-r-using-simple-features-package
model <- sf::st_read(dsn = gdb_path)

# Select Seamount, Seamount Ridges and Canyon
seamounts <- model$Feature[model$Feature %in% c("Seamount", "Seamount Ridges")]
canyons <- model$Feature[model$Feature %in% c("Canyon")]
others <- model$Feature[model$Feature %in% c("Coastal/Shelf Terrane", "Cross Shelf Valley",
                                            "Ridge", "Shelf Deep", "Trough Mouth Fan", 
                                            "Contourite Feature", "Plateau", "")]


In [ ]:
# List layers inside the geodatabase
gbd.layers <- st_layers(gdb_path)
print(gbd.layers)

In [ ]:
#st_point(c(1750160, 467499.9)) %>% # point coordinates
#  st_sfc(crs = st_crs(model))  # create feature collection, setting CRS to philly_sf's CRS


model_ctr <- st_point(c(1750160, 467499.9)) %>%
    st_sfc(crs = st_crs(model))
st_crs(model_ctr)$proj4string
model_buf <-  st_buffer(model_ctr, 10000)
model_sel <- st_filter(model, model_buf)


In [ ]:
model_intersection <- st_intersection(model_buf, model)
model_intersection

plot(st_geometry(model), border="#aaaaaa", main="Census tracts around city center,\nclipped by 2km buffer ")
plot(model_intersection, add=T, lwd = 2, border = "red")


In [ ]:
# If it's not WGS84 (EPSG:4326)

forms_to_coords <- function(model_sf){
    # data <- st_transform(model, 4326)
    # data <- st_make_valid(data)
    data <- st_make_valid(model_sf)
    
    # ⚠️ If geometry is NOT points
    #🔸 For polygons or lines
    centroids <- st_centroid(data)
    coords <- st_coordinates(centroids)
    return(coords)
}


# If it's not WGS84 (EPSG:4326)

forms_to_cs_coords <- function(model_sf, std.WGS84 = TRUE){
    coords <- st_transform(model_sf, 4326) %>%
        st_make_valid() %>%
        st_centroid() %>%
        st_coordinates()
    colnames(coords) <- c("lon","lat")
    return(coords)
}


forms_to_points <- function(model_sf){
    model_sf %>%
        st_transform(4326) %>%
        st_make_valid() %>%
        st_centroid()
}


In [ ]:
# Export to 
# write.csv(st_drop_geometry(data), paste("outputs/collection/st_drop_geometry.csv"), row.names = FALSE)

coords <- forms_to_coords(model)
head(coords, 5)

In [ ]:
# ==============================================================================
# DOWNLOAD AND LOAD BATHYMETRY
# ==============================================================================

options(timeout = 600)  # 10 minutes
bathy_file <- "global_topo_1min_topo_19_1.nc"
if(file.exists(bathy_file)){
    cat(bathy_file, "is (are) already in your repertory.")
} else {
    download.file("https://topex.ucsd.edu/pub/global_topo_1min/topo_19.1.nc", bathy_file, mode = "wb")
    print('File Downloaded')
}

# MOVE TO DATA DIRECTORY 
ifelse(!dir.exists(file.path(dataDir)),
        dir.create(file.path(dataDir)),
        "Directory Exists")

file.rename(from=bathy_file,
            to=paste(dataDir, bathy_file, sep = "/"))
bathy_file <- paste(dataDir, bathy_file, sep = "/")

In [ ]:
# Open the NetCDF file
nc_file <- nc_open(bathy_file)

# Get coordinates variables
dim_lon <- ncvar_get(nc_file, "lon", collapse_degen=FALSE)
dim_lat <- ncvar_get(nc_file, "lat", collapse_degen=FALSE)
dim_depth <- ncvar_get(nc_file, "z", collapse_degen=FALSE)
coords <- expand.grid(dim_lon, dim_lat)
depth_matrix <- data.frame(cbind(coords, as.vector(dim_depth)))
names(depth_matrix) <- c("lon", "lat", "depth")

nc_close(nc_file)


In [ ]:

# Filter Out the outerbound values
Antarctic_depth_coords <- depth_matrix %>%
  filter(
    lon >= 30, lon <= 150,
    lat >= -70, lat <= -60 #, depth <= 0
  )

In [ ]:
system ("conda install conda-forge::r-fnn")

library(FNN)


In [ ]:
# ==============================================================================
# CREATE GRID
# ==============================================================================
new.grid.lon <- seq(from=30.0, to=150.0, by=0.1)
new.grid.lat <- seq(from=-70.0, to=-60.0, by=0.1)
new.grid <- expand.grid(lon=new.grid.lon, lat=new.grid.lat)

new.grid$depth <- NA
# new.grid$geoFeat <- NA
# new.grid$canyonDist <- NA
# new.grid$seamountDist <- NA


grid <- as.data.table(new.grid)
depth_dt <- as.data.table(Antarctic_depth_coords)

nn <- get.knnx(
  data  = as.matrix(depth_dt[, .(lon, lat)]),
  query = as.matrix(grid[, .(lon, lat)]),
  k = 1
)
# Assign interpolated values
grid[, depth := depth_dt$depth[nn$nn.index]]

In [ ]:
# ==============================================================================
# ADD FEATURES TO THE GRID
# ==============================================================================

# Convert data (gdb layer) -> SpatVector → sf + Make sure polygons are valid
v_sf <- vect(gdb_path, layer=gbd.layers$name) %>%
    st_as_sf() %>%
    st_transform(4326) %>%
    st_make_valid()

# Ensure CRS is lon/lat + Convert your grid to sf points
grid_sf <- st_as_sf(
  grid,
  coords = c("lon", "lat"),
  crs = 4326
)

# Geometry issue : polygons in GDB geometrically invalid -> Disable s2
sf_use_s2(FALSE)

# Spatial join
grid_joined <- st_join(grid_sf, v_sf["Feature"], left = TRUE)
grid_joined <- grid_joined[!duplicated(st_coordinates(grid_joined)), ]

# Back to table
grid_final <- as.data.table(
  cbind(
    st_coordinates(grid_joined),
    st_drop_geometry(grid_joined)
  )
)

dim(grid_final)
# ================
# HANDLE NA VALUES
# ================

# Ensure correct column names
if(!all(c("lon","lat") %in% names(grid_final))){
  setnames(grid_final, c("X","Y"), c("lon","lat"))
}

# Identify NA rows
na_idx <- which(is.na(grid_final$Feature))

# Convert ONLY those points to sf
pts_na_sf <- st_as_sf(
  grid_final[na_idx],
  coords = c("lon","lat"),
  crs = 4326
)

# Find nearest polygon
nearest_idx <- st_nearest_feature(pts_na_sf, v_sf)

## Write back into grid_final
grid_final$Feature[na_idx] <- v_sf$Feature[nearest_idx]


In [ ]:
# Set binary data layers 
feature_map <- c(
  coastal.terrane    = "Coastal/Shelf Terrane",
  cross.shelf.valley = "Cross Shelf Valley",
  margin.ridges      = "Margin Ridges",
  shelf.bank         = "Bank",
  shelf.deep         = "Shelf Deep",
  trough.mouth.fan   = "Trough Mouth Fan",
  contourite.ft      = "Contourite Feature",
  plateau            = "Plateau",
  ridge.ft           = "Ridge"
)

for (col in names(feature_map)) {
  grid_final[[col]] <- as.integer(grid_final$Feature == feature_map[col])
}

In [ ]:
# ==============================================================================
# COMPUTE DISTANCES FROM THE CANYON AND SEAMOUNTS (FROM POLYGONS)
# ==============================================================================
compute_distance_to_feature <- function(grid, v_sf, feature_names, crs_proj = 3031) {

   grid_sf <- st_as_sf(
      grid_final,
      coords = c("lon", "lat"),
      crs = 4326
    )
  # 1. Select features
  feature_sf <- v_sf[v_sf$Feature %in% feature_names, ]
  
  # 2. Reproject
  grid_proj    <- st_transform(grid_sf, crs_proj)
  feature_proj <- st_transform(feature_sf, crs_proj)
  
  # 3. Find nearest feature
  nearest_idx <- st_nearest_feature(grid_proj, feature_proj)
  
  # 4. Compute distance
  dist <- st_distance(
    grid_proj,
    feature_proj[nearest_idx, ],
    by_element = TRUE
  )
  
  # 5. Return numeric vector (meters)
  return(as.numeric(dist))
}

In [ ]:
# Distance to seamounts
grid_final$dist_seamount <- compute_distance_to_feature(
  grid_final,
  v_sf,
  feature_names = c("Seamount", "Seamount Ridges")
)

# Distance to canyons
grid_final$dist_canyon <- compute_distance_to_feature(
  grid_final,
  v_sf,
  feature_names = c("Canyon")
)

# Truncate greater distances
# Convert to data.table if not already
setDT(grid_final)

grid_final <- grid_final %>%
  mutate(
    dist_seamount = pmin(dist_seamount, 50000),
    dist_canyon = ifelse(depth <= -2000,
                         pmin(dist_canyon, 100000),
                         dist_canyon)
  )



# Statistical Analyses

In [ ]:
# Default value of dissimilarity matrix

# -----------------------
# clean SST flag
# underwater.grid[sst == -32767, sst := NA]
# -----------------------

underwater.grid <- grid_final[grid_final$depth <= 0,]

rdm.rows <- sample(nrow(underwater.grid), 900, replace = FALSE)
X <- underwater.grid[rdm.rows, .(dist_seamount, depth, dist_canyon)]

# compute Gower distances (small subset!)
distance.matrix <- cluster::daisy(X, metric = "gower")
gower_matrix <- as.matrix(distance.matrix)

subset_clust <- c(200)
clara.object <- bioregion::nhclu_clara(
    dissimilarity = distance.matrix, n_clust = subset_clust)
                                       


In [ ]:
clusters900 <- underwater.grid[rdm.rows, ]
clusters900$K_200 <- clara.object$clusters$K_200

In [ ]:
summarise.n.cluster <- function(df, cluster.column){
    summarised.df <- df %>%
        # Here, we assume that EVERY k-cluster starts with "K_"
        mutate(across(starts_with("K_"), as.integer)) %>%
        group_by({{cluster.column}}) %>%

        # Create K-Centroids
        # Here, we assume that df contains sst, depth, iceC
        summarise(
            mean_dist_Smnt   = mean(dist_seamount, na.rm = TRUE),
            mean_depth = mean(depth, na.rm = TRUE),
            mean_dist_Cn  = mean(dist_canyon, na.rm = TRUE)
        )
    return(summarised.df)
}

upgma.classification <- function(df){
    data.table.var <- as.data.table(df)
    distance.matrix <- cluster::daisy(data.table.var, metric = "gower")
    hclust.K <- hclust(d = distance.matrix, method = "average")

    return(hclust.K)
    }

upgma.display.branch <- function(df, nb_clust = 12){
    hc <- upgma.classification(df)
    dend <- as.dendrogram(hc)
    upgma.cluster <- cutree(hc, k = nb_clust)
    cols <- rainbow(length(unique(upgma.cluster)))
    dend <- color_branches(dend, k = nb_clust, col = cols)
    plot(dend, cex = 0.6, main = paste("UPGMA Tree - Branches colored by",cutoff,"cluster(s)"))
}

upgma.display.rect <- function(df, nb_clust = 12){
    hc <- upgma.classification(df)
    plot(hc, hang = -1, labels = FALSE,cex = 0.6, main = paste("UPGMA Tree with",cutoff,"clusters"))
    rect.hclust(hc, k = nb_clust, border = rainbow(nb_clust))
    text(x = -0.05, y = 0.5, labels = 1:nb_clust, col = rainbow(nb_clust), lwd = 2, cex = 1.2)
}

infer.clust <- function(df = clusters900, custer.col = K_200, hclust.benthic = hclust.benthic.K200, cutoff = 12, x_name = "K_200"){
    X.1 <- summarise.n.cluster(df = df, cluster.column = {{custer.col}}) %>%
        rename(cluster_id = {{custer.col}})
    hclust.benthic$labels <- X.1$cluster_id
    
    groups <- cutree(hclust.benthic, k = cutoff)
    
    group_map <- data.frame(
      cluster_id = as.character(names(groups)),
      group_id = groups
    )
    
    df_with_groups <- merge(
      df,
      group_map,
      by.x = x_name,
      by.y = "cluster_id"
    )
    return(df_with_groups)
}



In [ ]:
df_K200 <- summarise.n.cluster(df = clusters900, cluster.column = K_200)
hclust.benthic.K200 <- upgma.classification(df_K200)

cutoff <- 12
plot(hclust.benthic.K200, hang = -1, cex = 0.6, main = paste("UPGMA Tree with",cutoff,"clusters"))
rect.hclust(hclust.benthic.K200, k = cutoff, border = seq(from = 1, to = 9, by = 1))

In [ ]:
clusters900_with_groups <- infer.clust(
    df = clusters900, custer.col = K_200,
    hclust.benthic = hclust.benthic.K200,
    cutoff = cutoff, x_name = "K_200")

k.centroids <- clusters900_with_groups %>%
  group_by(group_id) %>%
    summarise(
        mean_canDist   = mean(dist_canyon, na.rm = TRUE),
        mean_depth = mean(depth, na.rm = TRUE),
        mean_seamtDist  = mean(dist_seamount, na.rm = TRUE)
    )

# Centroids
centroids <- k.centroids[, c("mean_canDist", "mean_depth", "mean_seamtDist")]
colnames(centroids) <- c("dist_canyon", "depth", "dist_seamount")
# Grid data
grid_vals <- underwater.grid[, c("depth", "dist_canyon", "dist_seamount")]

# ------------------------------------------------------------------------------
# Combine for consistent scaling
scaled <- rbind(grid_vals, centroids) %>%
    scale()

grid_scaled <- scaled[1:nrow(grid_vals), ]
centroids_scaled <- scaled[(nrow(grid_vals)+1):nrow(scaled), ]

# Then run nearest neighbor
nn <- get.knnx(data = centroids_scaled, query = grid_scaled, k = 1)
underwater.grid$cluster <- nn$nn.index[,1]

export.csv(directory = outputDir, gower_matrix, "BenthicData_gower_K200.txt")


# Export GeoTIFF

In [ ]:
# Convert to data.frame (terra prefers that)
grid_df <- as.data.frame(underwater.grid)

# Create raster from XYZ (lon, lat, value)
r_canyon   <- rast(grid_df[, c("lon", "lat", "dist_canyon")], type = "xyz")
r_depth <- rast(grid_df[, c("lon", "lat", "depth")], type = "xyz")
r_seamount  <- rast(grid_df[, c("lon", "lat", "dist_seamount")], type = "xyz")
r_cluster <- rast(grid_df[, c("lon", "lat", "cluster")], type = "xyz")

r_stack <- c(r_canyon, r_depth, r_seamount, r_cluster)
crs(r_stack) <- "EPSG:4326"

names(r_stack) <- c("distCanyon", "depth", "distSeamount", "cluster")

In [ ]:
# ---- 1. Prepare data ----
grid_long <- grid_df %>%
  pivot_longer(
    cols = c(depth, dist_canyon, dist_seamount),  # rename if needed
    names_to = "variable",
    values_to = "value"
  ) %>%
  mutate(
    cluster = factor(cluster, levels = 1:12),   # adapt if not 12
    variable = recode(variable,
                      depth = "Depth (m)",
                      dist_canyon = "Distance to Canyons (m)",
                      dist_seamount = "Distance to Seamounts (m)")
  )

# ---- 2. Palette (consistent across everything) ----
colors <- c(
  "#E35C63", # red
  "#66E8FF", # blue
  "#3AA600", # green
  "#ACFD09", # lime green
  "#A70082", # plum
  "#FDFE72", # yellow
  "#AA65CE", # purple
  "#FBAA00", # orange
  "#79F4C6", # cyan
  "#D9C2FA", # lavender
  "#D4FEBF", # gin
  "#FEBCBD"  # pale pink
)
pal <- setNames(colors,
                levels(grid_long$cluster))

# ---- 3. Plot ----
p <- ggplot(grid_long,
            aes(x = value, y = cluster, fill = cluster)) +
  
  geom_violin(scale = "width", width = 0.8, color = "black", size = 0.2) +
  facet_wrap(~ variable, nrow = 1, scales = "free_x") +
  scale_fill_manual(values = pal) +
  labs(x = NULL, y = "Cluster number") +
  theme_minimal(base_size = 12) +
  theme(
    legend.position = "none",
    strip.text = element_text(face = "bold"),
    panel.spacing = unit(1.5, "lines")
  )

p

# ---- 4. Export ----
ggsave("outputs/collection/benthic_violin_panels.png", p, width = 12, height = 6, dpi = 300)

In [ ]:
# 1
png("outputs/collection/depth.png", width = 800, height = 600)
plot(r_stack[["depth"]], main = "Bathymetry")
dev.off()

# 2
png("outputs/collection/distSeamount.png", width = 800, height = 600)
plot(r_stack[["distSeamount"]], main = "Distance to Seamounts (m)")
dev.off()

# 3
png("outputs/collection/distCanyon.png", width = 800, height = 600)
plot(r_stack[["distCanyon"]], main = "Distance to Canyons (m)")
dev.off()

# Benthic bioregions
png("outputs/collection/BenBioregion.png", width = 800, height = 600)
plot(  r_stack[["cluster"]],  col = colors,
  breaks = seq(0.5, 12.5, by = 1),  # important for discrete classes
  main = "Benthic bioregions"
)
dev.off()

In [ ]:
writeRaster(
  r_stack,
  "outputs/collection/antarctic_benthic_environment.tif",
  filetype = "GTiff",
  overwrite = TRUE
)
